<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%205/5.3%20Task%20Flow%2C%20Memory%2C%20and%20Evaluation/5.3.3%20Tutorial_%20Error%20Handling%20and%20Response%20Validation%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###  Tutorial: Error Handling and Response Validation


In this tutorial, we'll learn how to add proper error handling to agents so they
don't crash when things go wrong. We'll start with a basic agent and upgrade it
with robust error handling.

Learning Objectives:
- Handle common agent errors gracefully
- Add try/catch blocks to agent tools
- Create fallback responses when tools fail
- Build agents that never crash

In [1]:
# Install required packages
!pip install -q "pydantic-ai-slim[openrouter]==2.48.0" openai==3.19.0 pydantic==2.13.5 python-dotenv==1.2.3

import os
from typing import List
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

load_dotenv()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 652.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.6/146.6 kB 2.8 MB/s eta 0:00:00


False

In [2]:
# OpenRouter setup
from getpass import getpass
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
MODEL = "openai/gpt-4.1-mini"

# PydanticAI model routed through OpenRouter; used by every agent below.
openrouter_model = OpenRouterModel(MODEL, provider=OpenRouterProvider(api_key=OPENROUTER_API_KEY))

Enter your OpenRouter API key: ··········


### Basic Agent (That Breaks)

Let's start with a simple agent that can crash.

In [3]:
basic_agent = Agent(openrouter_model)

In [4]:
@basic_agent.tool
async def breakable_search(ctx: RunContext[None], query: str) -> str:
    """Search tool that can fail."""
    import random
    if random.random() < 0.5:  # 50% chance of failure
        raise Exception("Network timeout!")
    return f"Search results for: {query}"

### Adding Try/Catch

Now let's fix it with error handling.

In [5]:
class RobustResponse(BaseModel):
    answer: str = Field(description="The main answer")
    had_errors: bool = Field(description="Whether there were errors")

robust_agent = Agent(openrouter_model, output_type=RobustResponse)

In [6]:
@robust_agent.tool
async def safe_search(ctx: RunContext[None], query: str) -> str:
    """Search tool with error handling."""
    try:
        import random
        if random.random() < 0.5:
            raise Exception("Network timeout!")
        return f"🔍 Search results for: {query}"
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return f"🔍 Search unavailable. Using general knowledge about: {query}"

In [7]:
@robust_agent.tool
async def safe_calculation(ctx: RunContext[None], expression: str) -> str:
    """Calculator with error handling."""
    try:
        # Simple validation
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            raise ValueError("Invalid characters")

        result = eval(expression)
        return f"🧮 {expression} = {result}"
    except ZeroDivisionError:
        return f"🧮 Cannot divide by zero in: {expression}"
    except Exception as e:
        return f"🧮 Cannot calculate: {expression} ({e})"

### Simple Error Tracking

Let's add basic error tracking.

In [8]:
error_count = 0

def track_error(operation: str, error: str):
    """Simple error tracking."""
    global error_count
    error_count += 1
    print(f"📊 Error #{error_count}: {operation} failed - {error}")

In [9]:
@robust_agent.tool
async def tracked_operation(ctx: RunContext[None], operation: str) -> str:
    """Operation with error tracking."""
    try:
        import random
        if random.random() < 0.3:
            raise Exception(f"{operation} service unavailable")

        return f"✅ {operation} completed successfully"
    except Exception as e:
        track_error(operation, str(e))
        return f"⚠️ {operation} failed, using fallback response"

In [10]:
test_cases = [
    "Search for Python tutorials",
    "Calculate 10 + 5",
    "Calculate 10 / 0"
]

for test in test_cases:
    print(f"\nTest: {test}")
    try:
        result = await robust_agent.run(test)
        print(f"✅ Success: {result.output.had_errors}")
        print(f"Answer: {result.output.answer[:80]}...")
    except Exception as e:
        print(f"❌ Failed: {e}")


Test: Search for Python tutorials
❌ Search failed: Network timeout!
❌ Search failed: Network timeout!
✅ Success: False
Answer: Here are some recommended Python tutorials for beginners in 2024:

1. Python.org...

Test: Calculate 10 + 5
✅ Success: False
Answer: 10 + 5 = 15...

Test: Calculate 10 / 0
✅ Success: True
Answer: Division by zero is undefined and cannot be performed....
